
<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Professional/main/Includes/images/deletes.png" width="60%">
</div>

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
from pyspark.sql import functions as F

schema = "customer_id STRING, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country_code STRING, row_status STRING, row_time timestamp"

(spark.readStream
        .table("bronze")
        .filter("topic = 'customers'")
        .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
        .select("v.*", F.col('v.row_time').alias("request_timestamp"))
        .filter("row_status = 'delete'")
        .select("customer_id", "request_timestamp",
                F.date_add("request_timestamp", 30).alias("deadline"), 
                F.lit("requested").alias("status"))
    .writeStream
        .outputMode("append")
        .option("checkpointLocation", "dbfs:/mnt/demo_pro/checkpoints/delete_requests")
        .trigger(availableNow=True)
        .table("delete_requests")
)

In [0]:
%sql
SELECT * FROM delete_requests

customer_id,request_timestamp,deadline,status
C00405,2022-03-07T21:20:00Z,2022-04-06,requested
C00420,2022-03-08T12:17:00Z,2022-04-07,requested
C00345,2022-03-09T09:38:00Z,2022-04-08,requested
C00300,2022-03-15T12:56:00Z,2022-04-14,requested
C00450,2022-03-16T18:18:00Z,2022-04-15,requested
C00390,2022-03-18T06:36:00Z,2022-04-17,requested
C00255,2022-03-18T16:12:00Z,2022-04-17,requested
C00525,2022-03-23T21:06:00Z,2022-04-22,requested
C00165,2022-03-23T22:42:00Z,2022-04-22,requested
C00225,2022-03-26T10:28:00Z,2022-04-25,requested


In [0]:
%sql
DELETE FROM customers_silver
WHERE customer_id IN (SELECT customer_id FROM delete_requests WHERE status = 'requested')

num_affected_rows
126


We previously enabled CDF in the customers table - we can leverage CDF data as incremental records of data changes to propogate deletes to downstream tables. We will first configure an incremental read of all change events committed to the customers silver table 

In [0]:
deleteDF = (spark.readStream
                 .format("delta")
                 .option("readChangeFeed", "true")
                 .option("startingVersion", 2)
                 .table("customers_silver"))

Define a function to be called before each batch to process the delete events. 

In [0]:
def process_deletes(microBatchDF, batchId):
    
    (microBatchDF
        .filter("_change_type = 'delete'")
        .createOrReplaceTempView("deletes"))

    microBatchDF._jdf.sparkSession().sql("""
        DELETE FROM customers_orders
        WHERE customer_id IN (SELECT customer_id FROM deletes)
    """)
    
    microBatchDF._jdf.sparkSession().sql("""
        MERGE INTO delete_requests r
        USING deletes d
        ON d.customer_id = r.customer_id
        WHEN MATCHED
          THEN UPDATE SET status = "deleted"
    """)

In [0]:
(deleteDF.writeStream
         .foreachBatch(process_deletes)
         .option("checkpointLocation", "dbfs:/mnt/demo_pro/checkpoints/deletes")
         .trigger(availableNow=True)
         .start())

In [0]:
%sql
SELECT * FROM delete_requests

customer_id,request_timestamp,deadline,status
C01110,2022-08-22T06:17:00Z,2022-09-21,deleted
C01095,2022-10-24T16:31:00Z,2022-11-23,deleted
C01080,2022-09-07T00:42:00Z,2022-10-07,deleted
C01065,2022-08-25T09:32:00Z,2022-09-24,deleted
C01050,2022-09-04T18:48:00Z,2022-10-04,deleted
C01035,2022-10-01T04:21:00Z,2022-10-31,deleted
C01020,2022-10-05T13:29:00Z,2022-11-04,deleted
C01005,2022-09-13T22:10:00Z,2022-10-13,deleted
C00990,2022-10-17T07:50:00Z,2022-11-16,deleted
C00975,2022-07-22T15:19:00Z,2022-08-21,deleted


We can take a look at the version history, and we will see that the latest version, we have performed a DELETE operation 

In [0]:
%sql
DESCRIBE HISTORY customers_orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2025-03-25T04:19:31Z,883594648125359,cwong2@deloitte.com.au,DELETE,"Map(predicate -> [""customer_id#1929 IN (list#1911 [])""])",null,List(3348297645639390),0122-053654-jmx9pij9,4,WriteSerializable,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 41071, numCopiedRows -> 516, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4270, numDeletedRows -> 41, scanTimeMs -> 3504, numAddedFiles -> 1, numAddedBytes -> 32363, rewriteTimeMs -> 727)",null,Databricks-Runtime/13.3.x-photon-scala2.12
4,2024-11-25T12:03:33Z,1984848388221464,rvemulapally@deloitte.com.au,DELETE,"Map(predicate -> [""customer_id#3724 IN (list#3706 [])""])",null,List(1183771821565958),0930-040814-iga72s2f,3,WriteSerializable,true,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 628, numDeletedRows -> 0, scanTimeMs -> 628, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/13.3.x-scala2.12
3,2024-11-21T04:39:51Z,8165668582725249,franyang@deloitte.com.au,MERGE,"Map(predicate -> [""((order_id#13565 = order_id#12638) AND (customer_id#13567 = customer_id#12640))""], matchedPredicates -> [{""predicate"":""(processed_timestamp#13579 < processed_timestamp#13534)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2162048479376614),0930-040814-iga72s2f,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 35709, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 116019, materializeSourceTimeMs -> 108280, numTargetRowsInserted -> 556, numTargetRowsMatchedDeleted -> 0, scanTimeMs -> 2919, numTargetRowsUpdated -> 0, numOutputRows -> 556, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 556, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2183)",null,Databricks-Runtime/13.3.x-scala2.12
2,2024-11-10T11:44:08Z,1984848388221464,rvemulapally@deloitte.com.au,MERGE,"Map(predicate -> [""((order_id#18765 = order_id#18022) AND (customer_id#18767 = customer_id#18024))""], matchedPredicates -> [{""predicate"":""(processed_timestamp#18779 < processed_timestamp#18734)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1183771821565987),0930-040814-iga72s2f,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 5362, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 55453, materializeSourceTimeMs -> 54315, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, scanTimeMs -> 406, numTargetRowsUpdated -> 0, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 680)",null,Databricks-Runtime/13.3.x-scala2.12
1,2024-11-10T11:41:25Z,1984848388221464,rvemulapally@deloitte.com.au,MERGE,"Map(predicate -> [""((order_id#13282 = order_id#12789) AND (customer_id#13284 = customer_id#12791))""], matchedPredicates -> [{""predicate"":""(processed_timestamp#13296 < processed_timestamp#13251)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""i

The following query will show only the records that were deleted in the previous version of the customers_orders table - we can see that the deleted records are still there. 

In [0]:
%sql
SELECT * FROM customers_orders@v4
EXCEPT
SELECT * FROM customers_orders

order_id,order_timestamp,customer_id,quantity,total,books,email,first_name,last_name,gender,street,city,country,row_time,processed_timestamp
000000006958,2022-07-08T11:05:00Z,C00945,1,41,"List(List(B08, 1, 41))",flyfedg@wikimedia.org,Fran,Lyfe,Female,29153 Mallard Circle,Hongyan,China,2022-07-08T10:29:00Z,2024-11-10T10:55:18Z
000000006988,2022-07-11T07:02:00Z,C00975,1,33,"List(List(B07, 1, 33))",null,Merrick,Malmar,Male,2877 Burning Wood Alley,Kalchevaya,Ukraine,2022-07-07T15:18:00Z,2024-11-10T10:55:18Z
000000007260,2022-08-07T18:03:00Z,C01065,1,24,"List(List(B09, 1, 24))",rmerrgenb3@uol.com.br,Romeo,Merrgen,Male,93705 Oriole Hill,Phichit,Thailand,2022-08-02T09:59:00Z,2024-11-10T10:55:18Z
000000006790,2022-06-26T04:09:00Z,C00900,1,47,"List(List(B05, 1, 47))",slembkedg@furl.net,Sarine,Lembke,Female,93 Rieder Place,Chincha Baja,Peru,2022-07-05T15:05:00Z,2024-11-10T10:55:18Z
000000007331,2022-08-15T22:02:00Z,C01110,1,24,"List(List(B09, 1, 24))",null,Lise,Mowsdell,Female,3178 Holmberg Court,Gaofeng,China,2022-08-11T06:30:00Z,2024-11-10T10:55:18Z
000000007327,2022-08-15T01:09:00Z,C01095,1,44,"List(List(B10, 1, 44))",nmoreby2z@ning.com,Nola,Moreby,Female,30008 Buell Place,Xingtan,China,2022-08-12T16:30:00Z,2024-11-10T10:55:18Z
000000007223,2022-08-03T11:03:00Z,C01065,1,24,"List(List(B09, 1, 24))",rmerrgenb3@uol.com.br,Romeo,Merrgen,Male,93705 Oriole Hill,Phichit,Thailand,2022-08-02T09:59:00Z,2024-11-10T10:55:18Z
000000006918,2022-07-05T03:09:00Z,C00960,1,24,"List(List(B09, 1, 24))",kmacariam@unesco.org,Katheryn,Macari,Female,9 Division Hill,Lenakapa,Indonesia,2022-07-20T03:07:00Z,2024-11-10T10:55:18Z
000000006985,2022-07-10T23:10:00Z,C00990,1,28,"List(List(B02, 1, 28))",null,Nicole,Martinovsky,Female,72443 Washington Plaza,Farranacoush,Ireland,2022-07-19T07:49:00Z,2024-11-10T10:55:18Z
000000007230,2022-08-04T11:04:00Z,C01080,1,47,"List(List(B05, 1, 47))",dmingardidz@gizmodo.com,Desdemona,Mingardi,Female,685 Montana Plaza,Krousón,Greece,2022-08-08T01:18:00Z,2024-11-10T10:55:18Z


This information is also still available in the CDF feed 

In [0]:
df = (spark.read
           .option("readChangeFeed", "true")
           .option("startingVersion", 2)
           .table("customers_silver")
           .filter("_change_type = 'delete'"))
display(df)

customer_id,email,first_name,last_name,gender,street,city,country,row_time,_change_type,_commit_version,_commit_timestamp
C01110,null,Lise,Mowsdell,Female,3178 Holmberg Court,Gaofeng,China,2022-08-11T06:30:00Z,delete,13,2025-03-25T04:11:52Z
C01095,nmoreby2z@ning.com,Nola,Moreby,Female,30008 Buell Place,Xingtan,China,2022-08-12T16:30:00Z,delete,13,2025-03-25T04:11:52Z
C01080,dmingardidz@gizmodo.com,Desdemona,Mingardi,Female,685 Montana Plaza,Krousón,Greece,2022-08-08T01:18:00Z,delete,13,2025-03-25T04:11:52Z
C01065,rmerrgenb3@uol.com.br,Romeo,Merrgen,Male,93705 Oriole Hill,Phichit,Thailand,2022-08-02T09:59:00Z,delete,13,2025-03-25T04:11:52Z
C01050,dmcshirie70@salon.com,Delora,McShirie,Female,89 Basil Drive,Nantes,France,2022-08-02T19:29:00Z,delete,13,2025-03-25T04:11:52Z
C01035,dmckeonfu@apple.com,Dugald,McKeon,Male,09 Milwaukee Crossing,Kraton,Indonesia,2022-07-28T04:20:00Z,delete,13,2025-03-25T04:11:52Z
C01020,lmccromleybo@soundcloud.com,Lucian,McCromley,Male,9 Cordelia Crossing,Chikwawa,Malawi,2022-08-01T14:48:00Z,delete,13,2025-03-25T04:11:52Z
C01005,dmatyushonokal@slideshare.net,Daphene,Matyushonok,Female,75 Karstens Plaza,Miharu,Japan,2022-07-19T22:09:00Z,delete,13,2025-03-25T04:11:52Z
C00990,null,Nicole,Martinovsky,Female,72443 Washington Plaza,Farranacoush,Ireland,2022-07-19T07:49:00Z,delete,13,2025-03-25T04:11:52Z
C00975,null,Merrick,Malmar,Male,2877 Burning Wood Alley,Kalchevaya,Ukraine,2022-07-07T15:18:00Z,delete,13,2025-03-25T04:11:52Z
